In [1]:
from azure.ai.ml import MLClient, command
from azure.identity import DefaultAzureCredential

# Connect
ml_client = MLClient(
    credential=DefaultAzureCredential(),
    subscription_id="e0b06bdd-434a-48ef-852a-4cfd218cc45e",
    resource_group_name="ml-learning-rg",
    workspace_name="my-azure-ml-workspace"
)
print(f"✅ Connected to: {ml_client.workspace_name}")

# Get correct environment
envs = list(ml_client.environments.list())
sklearn_env = [e for e in envs if 'sklearn-1.0' in e.name][0]
print(f"✅ Environment found: {sklearn_env.name}")

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


✅ Connected to: my-azure-ml-workspace
✅ Environment found: AzureML-sklearn-1.0-ubuntu20.04-py38-cpu


In [5]:
import shutil
import os

correct_path = '/home/azureuser/cloudfiles/code/Users/aazartaheri/Facebook_Ad_Campaign/'

# Create a job folder with BOTH the script AND the data
job_folder = f'{correct_path}job_package/'
os.makedirs(job_folder, exist_ok=True)

# Copy CSV to job folder
shutil.copy(f'{correct_path}KAG_conversion_data.csv', 
            f'{job_folder}KAG_conversion_data.csv')
print("✅ CSV copied to job folder")

# Write script using RELATIVE path (same folder)
script = """
import pandas as pd
import numpy as np
import json
import os

os.system("pip install scipy -q")
from scipy import stats

print("Starting A/B test analysis...")

# Use relative path - file is in same folder as script
df = pd.read_csv("KAG_conversion_data.csv")
print(f"Data loaded: {df.shape}")

df["campaign_group"] = df["xyz_campaign_id"].map({
    916: "A - Control",
    936: "B - Treatment 1",
    1178: "C - Treatment 2"
})

df["CTR"] = (df["Clicks"] / df["Impressions"] * 100).round(4)
df["conversion_rate"] = (df["Approved_Conversion"] / 
                          df["Impressions"] * 100).round(4)
df["ROAS"] = (df["Approved_Conversion"] / 
               df["Spent"].replace(0, np.nan)).round(4)

df_ab = df[df["campaign_group"].isin(
    ["A - Control", "B - Treatment 1"])].copy()
group_A = df_ab[df_ab["campaign_group"] == "A - Control"]
group_B = df_ab[df_ab["campaign_group"] == "B - Treatment 1"]
group_B_balanced = group_B.sample(n=len(group_A), random_state=42)

print(f"Group A: {len(group_A)} | Group B: {len(group_B_balanced)}")

_, p_conv = stats.ttest_ind(
    group_A["conversion_rate"],
    group_B_balanced["conversion_rate"]
)

lift = ((group_B_balanced["conversion_rate"].mean() -
         group_A["conversion_rate"].mean()) /
         group_A["conversion_rate"].mean() * 100)

results = {
    "run_date": str(pd.Timestamp.now()),
    "group_A_size": int(len(group_A)),
    "group_B_size": int(len(group_B_balanced)),
    "group_A_conversion_rate": round(float(group_A["conversion_rate"].mean()), 4),
    "group_B_conversion_rate": round(float(group_B_balanced["conversion_rate"].mean()), 4),
    "conversion_lift_pct": round(float(lift), 1),
    "p_value_conversion": round(float(p_conv), 4),
    "significant": bool(p_conv < 0.05),
    "winner": "B" if (p_conv < 0.05 and lift > 0) else "Inconclusive"
}

# Save results locally in job
with open("automated_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("="*40)
print("AUTOMATED A/B TEST RESULTS")
print("="*40)
print(f"Lift: {lift:.1f}%")
print(f"P-value: {p_conv:.4f}")
print(f"Winner: {results['winner']}")
print(f"Date: {results['run_date']}")
print("Analysis complete!")
"""

# Save script to job folder
with open(f'{job_folder}ab_test_script.py', 'w') as f:
    f.write(script)

print("✅ Job package ready!")
print(f"Contents of job folder:")
for f in os.listdir(job_folder):
    print(f"  📄 {f}")

✅ CSV copied to job folder
✅ Job package ready!
Contents of job folder:
  📄 ab_test_script.py
  📄 KAG_conversion_data.csv


In [6]:
from azure.ai.ml import command

job = command(
    name="ab_test_v6",
    display_name="Automated A/B Test v6",
    command="python ab_test_script.py",
    code=job_folder,  # Contains BOTH script AND CSV
    environment=f"{sklearn_env.name}@latest",
    compute="ml-compute",
    experiment_name="ab_test_automation"
)

print("🚀 Submitting...")
submitted_job = ml_client.jobs.create_or_update(job)
print(f"✅ Submitted: {submitted_job.name}")
print(f"Status: {submitted_job.status}")

🚀 Submitting...
✅ Submitted: ab_test_v6
Status: Starting


Uploading job_package (0.06 MBs): 100%|██████████| 62699/62699 [00:00<00:00, 1386038.70it/s]




In [7]:
import time, json, os

print("⏳ Monitoring...")
for i in range(20):
    time.sleep(15)
    job_status = ml_client.jobs.get(submitted_job.name)
    print(f"[{i*15}s] {job_status.status}")
    if job_status.status in ['Completed', 'Failed', 'Canceled']:
        break

if job_status.status == 'Completed':
    print(f"\n🎉 SUCCESS!")
    # Download outputs
    ml_client.jobs.download(
        name=submitted_job.name,
        download_path=f'{correct_path}job_outputs/',
        all=True
    )
    # Find and read results
    for root, dirs, files in os.walk(f'{correct_path}job_outputs/'):
        for file in files:
            if 'automated_results.json' in file:
                with open(os.path.join(root, file)) as f:
                    r = json.load(f)
                print(f"Winner: {r['winner']}")
                print(f"Lift: {r['conversion_lift_pct']}%")
                print(f"Date: {r['run_date']}")
else:
    print(f"\n❌ Failed")
    ml_client.jobs.download(
        name=submitted_job.name,
        download_path=f'{correct_path}logs_v6/',
        all=True
    )
    for root, dirs, files in os.walk(f'{correct_path}logs_v6/'):
        for file in files:
            if 'std_log' in file:
                with open(os.path.join(root, file), 'r', errors='ignore') as f:
                    print(f.read()[-2000:])

⏳ Monitoring...
[0s] Completed

🎉 SUCCESS!


In [10]:
import json
import os

print("📊 Final Automated Pipeline Results:")
print("="*40)

# Find results file
for root, dirs, files in os.walk(f'{correct_path}job_outputs/'):
    for file in files:
        if 'automated_results.json' in file:
            with open(os.path.join(root, file)) as f:
                r = json.load(f)
            print(f"Run Date:     {r['run_date']}")
            print(f"Group A Size: {r['group_A_size']}")
            print(f"Group B Size: {r['group_B_size']}")
            print(f"A Conv Rate:  {r['group_A_conversion_rate']}%")
            print(f"B Conv Rate:  {r['group_B_conversion_rate']}%")
            print(f"Lift:         {r['conversion_lift_pct']}%")
            print(f"P-value:      {r['p_value_conversion']}")
            print(f"Winner:       {r['winner']}")
            
            # Also save to main folder
            with open(f'{correct_path}automated_results.json', 'w') as f2:
                json.dump(r, f2, indent=2)
            print("\n✅ Results saved!")

📊 Final Automated Pipeline Results:
